# Offline GEO-filtered RAG over PDFs with LangChain + FAISS + Hugging Face + Qwen2.5-3B-Instruct (v2 threshold)

This notebook builds a local/offline RAG pipeline that:

- reads PDFs from `APAC`, `EMEA`, and `AMER` folders
- attaches GEO metadata to every page and chunk
- splits documents into chunks
- creates embeddings with a **local Hugging Face embedding model**
- builds **three separate FAISS indexes** (one per GEO)
- supports queries over **one GEO or multiple GEOs** like `APAC and AMER`
- retrieves only from the requested GEOs when the query mentions them
- applies a **retrieval score threshold** before calling the LLM
- uses **Qwen/Qwen2.5-3B-Instruct** as the **local/offline LLM**
- returns **"I do not have the answer in the provided documents."** when context is insufficient

## Why this model

`Qwen/Qwen2.5-3B-Instruct` is a much lighter local instruction model than `openai/gpt-oss-20b`,
and its model card documents standard Transformers usage and a long context window.
This makes it a more practical choice for a notebook-based local RAG prototype.

## What is new in this v2 step

This version adds threshold-based retrieval:

- retrieves chunks with FAISS scores
- keeps only chunks whose score passes the threshold
- returns no-answer when nothing passes
- includes a helper cell to inspect scores and tune the threshold on your own data

In [ ]:

# Install the core packages if needed
# Restart the kernel after installation if this is your first run.

# %pip install -qU langchain langchain-community langchain-text-splitters langchain-huggingface faiss-cpu pypdf sentence-transformers transformers accelerate torch tqdm



## Expected data layout

```text
data/
├── APAC/
│   ├── file1.pdf
│   └── file2.pdf
├── EMEA/
│   └── file3.pdf
└── AMER/
    └── file4.pdf
```

## Notes

- `PyPDFLoader` is best for text PDFs or PDFs that already contain a text layer.
- OCR fallback can be added later for scanned/image-only PDFs.
- The embedding model is separate from the chat model.
- This notebook uses the Transformers chat pipeline with message-style input.


In [1]:

from __future__ import annotations

import hashlib
import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

from tqdm.auto import tqdm

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from transformers import pipeline


In [2]:
@dataclass
class Settings:
    data_dir: Path = Path("data")
    storage_dir: Path = Path("storage")
    geos: Tuple[str, ...] = ("APAC", "EMEA", "AMER")

    # Chunking
    chunk_size: int = 800
    chunk_overlap: int = 100

    # Retrieval
    top_k: int = 3
    retrieval_fetch_k: int = 8
    score_threshold: Optional[float] = 1.2  # lower is stricter; tune on your data

    # Local embedding model
    embedding_model: str = "sentence-transformers/all-mpnet-base-v2"
    normalize_embeddings: bool = True

    # Local instruction model
    llm_model: str = "Qwen/Qwen2.5-3B-Instruct"

    # Generation
    max_new_tokens: int = 128
    do_sample: bool = False
    temperature: float = 0.0

    # Optional text quality guard
    min_page_chars: int = 40

    # Context trimming to keep memory use lower
    max_chars_per_chunk_in_prompt: int = 1200

settings = Settings()
settings.storage_dir.mkdir(parents=True, exist_ok=True)

settings

Settings(data_dir=WindowsPath('data'), storage_dir=WindowsPath('storage'), geos=('APAC', 'EMEA', 'AMER'), chunk_size=800, chunk_overlap=100, top_k=3, retrieval_fetch_k=8, score_threshold=1.2, embedding_model='sentence-transformers/all-mpnet-base-v2', normalize_embeddings=True, llm_model='Qwen/Qwen2.5-3B-Instruct', max_new_tokens=128, do_sample=False, temperature=0.0, min_page_chars=40, max_chars_per_chunk_in_prompt=1200)

## Helper functions

In [3]:

def normalize_geo(value: str) -> str:
    value = value.strip().upper()
    if value not in settings.geos:
        raise ValueError(f"Unsupported GEO: {value}")
    return value


def detect_geos_from_query(query: str) -> List[str]:
    text = query.upper()
    found = []

    for geo in settings.geos:
        if re.search(rf"\b{geo}\b", text):
            found.append(geo)

    return found


def make_doc_id(path: Path) -> str:
    return hashlib.md5(str(path.resolve()).encode("utf-8")).hexdigest()


def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def page_has_usable_text(text: str, min_chars: int = settings.min_page_chars) -> bool:
    text = clean_text(text)
    return len(text) >= min_chars


def build_chunk_id(source: str, page: Optional[int], idx: int) -> str:
    raw = f"{source}|{page}|{idx}"
    return hashlib.md5(raw.encode("utf-8")).hexdigest()


## 1) Load PDFs and attach metadata

In [4]:

def load_geo_documents(data_dir: Path, geo: str) -> List[Document]:
    geo = normalize_geo(geo)
    geo_dir = data_dir / geo
    if not geo_dir.exists():
        print(f"Warning: {geo_dir} does not exist")
        return []

    all_docs: List[Document] = []
    pdf_paths = sorted(geo_dir.rglob("*.pdf"))

    for pdf_path in tqdm(pdf_paths, desc=f"Loading {geo} PDFs"):
        loader = PyPDFLoader(str(pdf_path))
        pages = loader.load()

        doc_id = make_doc_id(pdf_path)

        for page_doc in pages:
            raw_text = page_doc.page_content or ""
            cleaned = clean_text(raw_text)

            if not page_has_usable_text(cleaned):
                # Later you can replace this with OCR fallback.
                continue

            metadata = dict(page_doc.metadata)
            metadata.update(
                {
                    "geo": geo,
                    "source": str(pdf_path),
                    "doc_id": doc_id,
                    "file_name": pdf_path.name,
                    "page": metadata.get("page"),
                }
            )

            all_docs.append(
                Document(
                    page_content=cleaned,
                    metadata=metadata,
                )
            )

    return all_docs


In [5]:

geo_page_docs: Dict[str, List[Document]] = {}

for geo in settings.geos:
    docs = load_geo_documents(settings.data_dir, geo)
    geo_page_docs[geo] = docs
    print(f"{geo}: {len(docs)} usable page documents")


Loading APAC PDFs:   0%|          | 0/2 [00:00<?, ?it/s]

APAC: 116 usable page documents


Loading EMEA PDFs:   0%|          | 0/2 [00:00<?, ?it/s]

EMEA: 79 usable page documents


Loading AMER PDFs:   0%|          | 0/1 [00:00<?, ?it/s]

AMER: 17 usable page documents


## 2) Split into chunks

In [6]:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=settings.chunk_size,
    chunk_overlap=settings.chunk_overlap,
)


def chunk_documents(docs: List[Document]) -> List[Document]:
    chunks = text_splitter.split_documents(docs)

    for idx, chunk in enumerate(chunks):
        source = chunk.metadata.get("source", "")
        page = chunk.metadata.get("page")
        chunk.metadata["chunk_id"] = build_chunk_id(source, page, idx)

    return chunks


In [7]:

geo_chunk_docs: Dict[str, List[Document]] = {}

for geo in settings.geos:
    chunks = chunk_documents(geo_page_docs[geo])
    geo_chunk_docs[geo] = chunks
    print(f"{geo}: {len(chunks)} chunks")


APAC: 157 chunks
EMEA: 161 chunks
AMER: 43 chunks


## 3) Create local embeddings

This version normalizes embeddings before FAISS indexing.
That makes distance-based threshold tuning more stable across runs.

In [8]:
embeddings = HuggingFaceEmbeddings(
    model_name=settings.embedding_model,
    encode_kwargs={"normalize_embeddings": settings.normalize_embeddings},
)
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2', cache_folder=None, model_kwargs={}, encode_kwargs={'normalize_embeddings': True}, query_encode_kwargs={}, multi_process=False, show_progress=False)

## 4) Build and save one FAISS index per GEO

In [9]:

def geo_index_path(geo: str) -> Path:
    return settings.storage_dir / f"faiss_{geo.lower()}"


def build_and_save_geo_indexes(
    geo_to_chunks: Dict[str, List[Document]],
    embeddings_model: HuggingFaceEmbeddings,
) -> Dict[str, FAISS]:
    stores: Dict[str, FAISS] = {}

    for geo, chunks in geo_to_chunks.items():
        if not chunks:
            print(f"Skipping {geo}: no chunks found")
            continue

        print(f"Building FAISS index for {geo} with {len(chunks)} chunks...")
        vectorstore = FAISS.from_documents(chunks, embeddings_model)

        save_dir = geo_index_path(geo)
        save_dir.mkdir(parents=True, exist_ok=True)
        vectorstore.save_local(str(save_dir))

        stores[geo] = vectorstore

    return stores


In [10]:

# Run this once to build the indexes
geo_vectorstores = build_and_save_geo_indexes(geo_chunk_docs, embeddings)


Building FAISS index for APAC with 157 chunks...
Building FAISS index for EMEA with 161 chunks...
Building FAISS index for AMER with 43 chunks...


## 5) Load saved FAISS indexes

In [11]:

def load_geo_indexes(embeddings_model: HuggingFaceEmbeddings) -> Dict[str, FAISS]:
    stores: Dict[str, FAISS] = {}

    for geo in settings.geos:
        save_dir = geo_index_path(geo)
        if save_dir.exists():
            stores[geo] = FAISS.load_local(
                str(save_dir),
                embeddings_model,
                allow_dangerous_deserialization=True,
            )
    return stores


# Uncomment in a fresh session after indexes already exist:
# geo_vectorstores = load_geo_indexes(embeddings)


## 6) Load the local instruction model

In [12]:

generator = pipeline(
    "text-generation",
    model=settings.llm_model,
    torch_dtype="auto",
    device_map="auto",
)

generator


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


## 7) Retrieval and answer generation

This version supports:

- single GEO: `APAC`
- multiple GEOs: `APAC and AMER`
- no GEO mentioned: searches all available GEO indexes
- score-threshold filtering before the LLM is called

**Important:** with FAISS distances here, **lower score is better**.
Use the score inspection helper below to tune `settings.score_threshold` for your data.

In [13]:
def format_context(docs: List[Document]) -> str:
    blocks = []
    for i, doc in enumerate(docs[: settings.top_k], start=1):
        source = doc.metadata.get("file_name", "unknown")
        page = doc.metadata.get("page", "unknown")
        geo = doc.metadata.get("geo", "unknown")
        text = doc.page_content[: settings.max_chars_per_chunk_in_prompt]
        blocks.append(
            f"[Chunk {i}]\n"
            f"GEO: {geo}\n"
            f"Source: {source}\n"
            f"Page: {page}\n"
            f"Content:\n{text}"
        )
    return "\n\n---\n\n".join(blocks)


def inspect_retrieval_scores(
    question: str,
    stores: Dict[str, FAISS],
    k: int = 5,
) -> List[Dict[str, object]]:
    requested_geos = detect_geos_from_query(question)
    target_geos = requested_geos if requested_geos else list(stores.keys())

    rows = []

    for geo in target_geos:
        if geo not in stores:
            continue

        docs_and_scores = stores[geo].similarity_search_with_score(question, k=k)
        for doc, score in docs_and_scores:
            rows.append(
                {
                    "geo": geo,
                    "score": float(score),
                    "file_name": doc.metadata.get("file_name"),
                    "page": doc.metadata.get("page"),
                    "chunk_id": doc.metadata.get("chunk_id"),
                    "preview": doc.page_content[:180].replace("\n", " "),
                }
            )

    rows.sort(key=lambda x: x["score"])
    return rows


def retrieve_documents(
    question: str,
    stores: Dict[str, FAISS],
    k: int = settings.top_k,
    fetch_k: int = settings.retrieval_fetch_k,
    score_threshold: Optional[float] = None,
) -> Tuple[List[str], List[Document], List[Tuple[Document, float]]]:
    requested_geos = detect_geos_from_query(question)
    target_geos = requested_geos if requested_geos else list(stores.keys())
    threshold = settings.score_threshold if score_threshold is None else score_threshold

    scored_results: List[Tuple[Document, float]] = []

    for geo in target_geos:
        if geo not in stores:
            continue

        docs_and_scores = stores[geo].similarity_search_with_score(question, k=fetch_k)
        for doc, score in docs_and_scores:
            scored_results.append((doc, float(score)))

    scored_results.sort(key=lambda x: x[1])  # lower score is better

    filtered_results = []
    seen_keys = set()

    for doc, score in scored_results:
        if threshold is not None and score > threshold:
            continue

        key = (
            doc.metadata.get("file_name"),
            doc.metadata.get("page"),
            doc.metadata.get("chunk_id"),
        )
        if key in seen_keys:
            continue

        seen_keys.add(key)
        filtered_results.append((doc, score))

        if len(filtered_results) >= k:
            break

    final_docs = [doc for doc, _ in filtered_results]
    return target_geos, final_docs, filtered_results

In [14]:
SYSTEM_INSTRUCTION = (
    "You are a retrieval-augmented assistant. "
    "Answer only from the supplied context. "
    "Do not use outside knowledge. "
    "If the answer is not present in the context, reply exactly with: "
    "'I do not have the answer in the provided documents.'"
)


def extract_assistant_text(generated_output) -> str:
    generated = generated_output[0]["generated_text"]

    if isinstance(generated, list):
        last_item = generated[-1]
        if isinstance(last_item, dict):
            content = last_item.get("content", "")
            if isinstance(content, str):
                return content.strip()

    return str(generated).strip()


def answer_question(question: str, stores: Dict[str, FAISS]) -> Dict[str, object]:
    geos, docs, scored_docs = retrieve_documents(question, stores)

    if not docs:
        return {
            "question": question,
            "geo_used": geos,
            "score_threshold": settings.score_threshold,
            "answer": "I do not have the answer in the provided documents.",
            "sources": [],
        }

    context = format_context(docs)
    geo_label = ", ".join(geos) if geos else "ALL"

    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {
            "role": "user",
            "content": (
                f"GEO filter: {geo_label}\n\n"
                f"Context:\n{context}\n\n"
                f"Question: {question}"
            ),
        },
    ]

    outputs = generator(
        messages,
        max_new_tokens=settings.max_new_tokens,
        do_sample=settings.do_sample,
        temperature=settings.temperature,
    )

    answer = extract_assistant_text(outputs)

    sources = [
        {
            "geo": doc.metadata.get("geo"),
            "file_name": doc.metadata.get("file_name"),
            "page": doc.metadata.get("page"),
            "chunk_id": doc.metadata.get("chunk_id"),
            "score": score,
        }
        for doc, score in scored_docs
    ]

    return {
        "question": question,
        "geo_used": geos,
        "score_threshold": settings.score_threshold,
        "answer": answer,
        "sources": sources,
    }

## 8) Inspect retrieval scores first

Before tuning the threshold, inspect a few retrieval scores on your own questions.

Start with a sample query, review the returned scores, then adjust:

```python
settings.score_threshold = 1.2
```

Lower values are stricter. If good answers disappear, raise the threshold slightly.

In [15]:
score_rows = inspect_retrieval_scores("What is the refund policy for APAC?", geo_vectorstores, k=5)
for row in score_rows[:5]:
    print(json.dumps(row, indent=2))

{
  "geo": "APAC",
  "score": 1.728743314743042,
  "file_name": "5deb6b48-5d7f-4a37-a361-d01317ac4bb7.pdf",
  "page": 4,
  "chunk_id": "019449a57cc1c02e04b70ac5a11f7367",
  "preview": "all positions in the decoder up to and including that position. We need to prevent leftward information \ufb02ow in the decoder to preserve the auto-regressive property. We implement th"
}
{
  "geo": "APAC",
  "score": 1.7624207735061646,
  "file_name": "5deb6b48-5d7f-4a37-a361-d01317ac4bb7.pdf",
  "page": 9,
  "chunk_id": "87be65be9c289df767ef6658a018d3e6",
  "preview": "arXiv:1308.0850, 2013. [10] Kaiming He, Xiangyu Zhang, Shaoqing Ren, and Jian Sun. Deep residual learning for im- age recognition. In Proceedings of the IEEE Conference on Computer"
}
{
  "geo": "APAC",
  "score": 1.797020435333252,
  "file_name": "5deb6b48-5d7f-4a37-a361-d01317ac4bb7.pdf",
  "page": 4,
  "chunk_id": "64f2cb14064f18a3b608ae9ea8f42363",
  "preview": "3.5 Positional Encoding Since our model contains no recurrence and no c

## 9) Example queries

In [16]:
result = answer_question("What is the refund policy for APAC?", geo_vectorstores)
print(result["answer"])
print(json.dumps(result["sources"][:3], indent=2))

I do not have the answer in the provided documents.
[]


In [17]:
result = answer_question("Compare the refund policy for APAC and AMER", geo_vectorstores)
print(result["answer"])
print(json.dumps(result["sources"][:5], indent=2))

I do not have the answer in the provided documents.
[]


In [18]:
result = answer_question("What are the onboarding requirements?", geo_vectorstores)
print(result["answer"])
print(json.dumps(result["sources"][:3], indent=2))

I do not have the answer in the provided documents.
[]


## 10) If memory is still tight

Try these settings:

```python
settings.top_k = 2
settings.retrieval_fetch_k = 5
settings.max_new_tokens = 64
settings.max_chars_per_chunk_in_prompt = 800
```

If needed, you can also switch the model to `Qwen/Qwen2.5-1.5B-Instruct`.

## 11) Summary

This notebook keeps your original RAG design and adds the first v2 improvement:

- Loader: `PyPDFLoader`
- Splitter: `RecursiveCharacterTextSplitter`
- Embeddings: `HuggingFaceEmbeddings` with normalization
- Vector store: `FAISS`
- LLM: `Qwen/Qwen2.5-3B-Instruct`
- GEO filter: one FAISS index per GEO
- Query support: single GEO, multiple GEOs, or all GEOs
- New v2 feature: score-threshold filtering before generation